
# 🤖 Notebook 2 — Machine Learning interpretable para metilación en OPMD

**Inspirado en:** Buenahora, Lafaurie & Perdomo  
**Artículo:** https://pmc.ncbi.nlm.nih.gov/articles/PMC8451454/

---

## Objetivo metodológico

Construir un pipeline de **Machine Learning interpretable y sin fuga de información** para datos epigenómicos de alta dimensionalidad.

El énfasis no será “buscar el accuracy más alto”, sino responder:

> ¿Cómo construir una estrategia reproducible para seleccionar biomarcadores y clasificar perfiles moleculares cuando tenemos muchísimas variables y muy pocas muestras?

### ⚠️ Advertencia

Este notebook usa **datos sintéticos didácticos**.  
Con `n = 12`, cualquier métrica predictiva es altamente inestable y **no representa rendimiento clínico**.

El objetivo es mostrar el método que después puede transferirse a una cohorte real y ampliada.


In [ ]:

# ============================================================
# 0. Configuración
# ============================================================
import sys, subprocess, pkgutil

packages = ["numpy", "pandas", "matplotlib", "scipy", "sklearn"]
for p in packages:
    if pkgutil.find_loader(p) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold, RepeatedStratifiedKFold, cross_validate,
    permutation_test_score
)
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, balanced_accuracy_score,
    f1_score
)

SEED = 2026
rng = np.random.default_rng(SEED)
print("Entorno listo.")



## 1. Problema `p >> n`

En datos epigenómicos es normal tener:

- **miles o millones de características**;
- **decenas o cientos de pacientes**.

Esto crea riesgo de:

- sobreajuste;
- selección espuria de biomarcadores;
- fuga de información;
- métricas excesivamente optimistas.

### Regla metodológica central

> Toda selección de características debe ocurrir **dentro** del proceso de validación cruzada.

Por eso utilizaremos `Pipeline` de scikit-learn.


In [ ]:

# ============================================================
# 1. Datos sintéticos con 3 clases
# ============================================================
groups = np.array(
    ["p16+_HPV16+"] * 4 +
    ["p16+_HPV16-"] * 4 +
    ["p16-_HPV16-"] * 4
)
samples = np.array([f"S{i:02d}" for i in range(1, 13)])

n_features = 3000
feature_names = np.array([f"CpG_SIM_{i:05d}" for i in range(1, n_features + 1)])

base = rng.beta(2.2, 2.2, size=n_features)
X = np.tile(base, (12, 1)) + rng.normal(0, 0.06, size=(12, n_features))

# Señales didácticas
for i, g in enumerate(groups):
    if g.startswith("p16+"):
        X[i, :100] += 0.16
    if g == "p16+_HPV16+":
        X[i, 100:180] += 0.20
        X[i, 180:240] -= 0.18

X = np.clip(X, 0, 1)
y = groups.copy()

print("X:", X.shape)
print("Clases:", pd.Series(y).value_counts().to_dict())



## 2. Modelo principal

Usaremos una **regresión logística multinomial regularizada**, precedida por:

1. selección univariada de las `k` características más informativas;
2. estandarización;
3. regularización L2.

¿Por qué este modelo?

- es interpretable;
- funciona como baseline fuerte;
- su complejidad es controlable;
- permite inspeccionar coeficientes;
- es más razonable que una red neuronal con 12 muestras.

En una tesis real, el mismo principio podría evolucionar hacia Elastic Net, modelos de supervivencia penalizados o aprendizaje multimodal.


In [ ]:

# ============================================================
# 2. Pipeline sin fuga
# ============================================================
pipe = Pipeline([
    ("select", SelectKBest(score_func=f_classif, k=40)),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        C=0.5,
        solver="lbfgs",
        max_iter=5000,
        random_state=SEED
    ))
])

pipe



## 3. Validación cruzada repetida

Con 4 observaciones por clase, utilizaremos validación estratificada de 2 folds repetida varias veces.

Esto sigue siendo un experimento **muy inestable**. El objetivo es observar variabilidad, no vender una única cifra.


In [ ]:

# ============================================================
# 3. CV repetida
# ============================================================
cv = RepeatedStratifiedKFold(
    n_splits=2,
    n_repeats=20,
    random_state=SEED
)

scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro"
}

cv_res = cross_validate(
    pipe,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

summary = pd.DataFrame({
    "métrica": ["Balanced accuracy", "F1 macro"],
    "media_CV": [
        cv_res["test_balanced_accuracy"].mean(),
        cv_res["test_f1_macro"].mean()
    ],
    "desv_std_CV": [
        cv_res["test_balanced_accuracy"].std(),
        cv_res["test_f1_macro"].std()
    ],
    "media_train": [
        cv_res["train_balanced_accuracy"].mean(),
        cv_res["train_f1_macro"].mean()
    ]
})

display(summary.round(3))


In [ ]:

# Visualizar distribución de métricas
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.boxplot(
    [cv_res["test_balanced_accuracy"], cv_res["test_f1_macro"]],
    tick_labels=["Balanced accuracy", "F1 macro"]
)
ax.axhline(1/3, linestyle="--", linewidth=1, label="Referencia azar aprox. (3 clases)")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Variabilidad del rendimiento en CV repetida")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()



### Cómo interpretar esto

No basta con decir:

> “El modelo obtuvo X%.”

Debemos preguntar:

- ¿qué tan dispersa es la métrica?
- ¿hay diferencia fuerte entre entrenamiento y validación?
- ¿cambia mucho según el split?
- ¿el resultado supera lo esperable por azar?
- ¿se mantiene en una cohorte externa?

Con 12 muestras, estas preguntas son más importantes que el valor puntual.



## 4. Prueba de permutación

La prueba de permutación ayuda a preguntar:

> Si las etiquetas no tuvieran relación real con los perfiles moleculares, ¿con qué frecuencia obtendríamos un score igual o mejor por azar?

Esto no resuelve el problema del tamaño muestral, pero es una comprobación útil.


In [ ]:

# ============================================================
# 4. Permutation test
# ============================================================
cv_perm = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)

score, perm_scores, pvalue = permutation_test_score(
    pipe,
    X,
    y,
    scoring="balanced_accuracy",
    cv=cv_perm,
    n_permutations=200,
    n_jobs=-1,
    random_state=SEED
)

print(f"Balanced accuracy observada: {score:.3f}")
print(f"p-valor de permutación:       {pvalue:.4f}")

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.hist(perm_scores, bins=20, alpha=0.75)
ax.axvline(score, linewidth=2, label=f"Observado = {score:.3f}")
ax.set_xlabel("Balanced accuracy bajo permutación")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución nula por permutación")
ax.legend()
plt.tight_layout()
plt.show()



## 5. Predicciones out-of-fold para inspección

Construiremos predicciones en un esquema estratificado simple para producir una matriz de confusión ilustrativa.

**Importante:** esta matriz sigue siendo exploratoria.


In [ ]:

# ============================================================
# 5. Predicciones out-of-fold manuales
# ============================================================
cv_oof = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
pred = np.empty_like(y, dtype=object)

for train_idx, test_idx in cv_oof.split(X, y):
    model = clone(pipe)
    model.fit(X[train_idx], y[train_idx])
    pred[test_idx] = model.predict(X[test_idx])

print(classification_report(y, pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y, pred,
    xticks_rotation=25
)
plt.title("Matriz de confusión OOF — demostración")
plt.tight_layout()
plt.show()



## 6. Interpretabilidad: biomarcadores candidatos

Entrenaremos el pipeline con todas las muestras **solo para inspección exploratoria**, no para estimar rendimiento.

Extraeremos:

- características seleccionadas;
- coeficientes por clase;
- ranking por magnitud absoluta.

En un estudio real, un biomarcador candidato debería mostrar además:

- estabilidad entre remuestreos;
- coherencia biológica;
- validación externa;
- idealmente validación experimental.


In [ ]:

# ============================================================
# 6. Coeficientes del modelo ajustado
# ============================================================
final_model = clone(pipe).fit(X, y)

selector = final_model.named_steps["select"]
selected_mask = selector.get_support()
selected_features = feature_names[selected_mask]

coef = final_model.named_steps["model"].coef_
classes = final_model.named_steps["model"].classes_

coef_df = pd.DataFrame(
    coef.T,
    index=selected_features,
    columns=classes
)

coef_df["max_abs_coef"] = coef_df.abs().max(axis=1)
coef_df = coef_df.sort_values("max_abs_coef", ascending=False)

display(coef_df.head(20).round(3))


In [ ]:

# Top características
top = coef_df.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top.index, top["max_abs_coef"])
ax.set_xlabel("Máximo |coeficiente| entre clases")
ax.set_title("Top características seleccionadas — interpretación exploratoria")
plt.tight_layout()
plt.show()



## 7. Estabilidad de selección

Un problema frecuente en `p >> n` es que distintas muestras producen distintos “biomarcadores importantes”.

Evaluaremos cuántas veces aparece cada característica dentro de las 40 seleccionadas a través de múltiples remuestreos.

La **frecuencia de selección** es más informativa que un único ranking.


In [ ]:

# ============================================================
# 7. Frecuencia de selección
# ============================================================
cv_stability = RepeatedStratifiedKFold(
    n_splits=2,
    n_repeats=40,
    random_state=SEED
)

counts = pd.Series(0, index=feature_names, dtype=int)
n_fits = 0

for train_idx, test_idx in cv_stability.split(X, y):
    temp = clone(pipe).fit(X[train_idx], y[train_idx])
    mask = temp.named_steps["select"].get_support()
    counts.loc[feature_names[mask]] += 1
    n_fits += 1

stability = (counts / n_fits).sort_values(ascending=False)
stability_df = stability.head(30).rename("selection_frequency").to_frame()

display(stability_df.head(20).round(3))


In [ ]:

fig, ax = plt.subplots(figsize=(8, 7))
tmp = stability_df.head(20).iloc[::-1]
ax.barh(tmp.index, tmp["selection_frequency"])
ax.set_xlim(0, 1)
ax.set_xlabel("Frecuencia de selección")
ax.set_title("Estabilidad de biomarcadores candidatos")
plt.tight_layout()
plt.show()



## 8. Qué NO deberíamos hacer

### ❌ Error 1 — Seleccionar variables antes de CV
Seleccionar los CpGs usando todos los datos y luego hacer cross-validation produce **data leakage**.

### ❌ Error 2 — Reportar accuracy solamente
Con clases pequeñas o desbalanceadas, deben considerarse métricas como:
- balanced accuracy;
- macro-F1;
- sensibilidad/especificidad por clase.

### ❌ Error 3 — Usar Deep Learning con 12 muestras
Una arquitectura compleja puede memorizar el dataset.

### ❌ Error 4 — Interpretar asociación como causalidad
Un CpG asociado con una clase no implica que cause la enfermedad.

### ❌ Error 5 — Validar en el mismo conjunto
Para una aplicación clínica se necesita cohorte independiente.



## 9. Evolución hacia una tesis doctoral

Una estrategia doctoral sólida podría ampliar este pipeline hacia:

### A. Descubrimiento de biomarcadores
- Elastic Net / Sparse Group Lasso
- estabilidad de selección
- integración de anotación biológica

### B. Predicción longitudinal
Si existe seguimiento:
- Cox penalizado
- Random Survival Forest
- Gradient Boosting para supervivencia
- DeepSurv cuando el tamaño muestral lo justifique

### C. Integración multimodal
Combinar:
- metilación;
- variables clínicas;
- histopatología;
- transcriptómica;
- estado HPV/p16.

### D. Interpretabilidad
- coeficientes penalizados;
- SHAP cuando sea apropiado;
- importancia por permutación;
- estabilidad por bootstrap;
- análisis de vías.

### E. Validación
1. interna;
2. externa;
3. temporal;
4. multicéntrica.



## 10. Pregunta doctoral que emerge

> ¿Es posible identificar una firma molecular estable, interpretable y externamente validable que combine metilación del ADN con variables clínicas para estimar el riesgo de progresión de OPMD?

Esta formulación es metodológicamente más fuerte que preguntar simplemente:

> “¿Qué algoritmo clasifica mejor?”

porque centra la investigación en:

- reproducibilidad;
- biomarcadores;
- generalización;
- relevancia clínica.



## 11. Qué datos pedir a la Dra. Sandra

Para convertir este notebook en un análisis real sería ideal disponer de:

- matriz CpG/DMR × muestra;
- ubicación genómica;
- cobertura;
- p16;
- HPV16;
- diagnóstico histopatológico;
- edad/sexo/localización;
- lote de procesamiento;
- tratamiento previo;
- seguimiento;
- fecha/evento de progresión si existe.

### Variable objetivo prioritaria si existe seguimiento

`tiempo hasta transformación maligna`

Esto permitiría pasar de clasificación transversal a **modelos de supervivencia y riesgo**, mucho más alineados con una aplicación clínica.



# Conclusión

Este notebook propone una columna vertebral metodológica:

**datos ómicos → selección dentro de CV → regularización → evaluación robusta → prueba contra azar → estabilidad → interpretabilidad → validación externa**

La idea central para una conversación doctoral es:

> El objetivo no es usar IA por usar IA. El objetivo es encontrar señales moleculares reproducibles que puedan responder una pregunta clínica y sobrevivir a validación externa.
